<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/pandas/projects/MedCare_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Toshkentdagi xususiy klinika bizni yolladi. Ularning 3 yillik bemorlar ma'lumotlari bor, lekin hech qanday tahlil qilinmagan. Bizning vazifamiz — bu ma'lumotlardan amaliy biznes qarorlar chiqarish**

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

- *Dataset sun'iy intellektdan olindi*

In [2]:
df = pd.read_csv("/content/drive/MyDrive/AI learning/pandas_2026/projects/medcare_dataset.csv")
df.head()

,bemor_id,ism,yosh,jinsi,viloyat,bo_lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
0,P00001,Bemor_1,52,Ayol,Farg'ona,Nevrologiya,Dr. Rahimov S,Migren,2022-02-20,14.0,2999095.0,Karta,Og'ir,True,1.0
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,1465502.0,Naqd,Surunkali,True,3.0
2,P00003,Bemor_3,72,Erkak,Surxondaryo,Pediatriya,Dr. Tursunova G,Ich ketish,2023-03-28,23.0,4530088.0,Naqd,Surunkali,False,3.0
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,1349110.0,Sug'urta,Yaxshilandi,True,4.0
4,P00005,Bemor_5,21,Erkak,Farg'ona,Jarrohlik,Dr. Abdullayev K,O'tkir appenditsit,2024-01-02,7.0,996462.0,Naqd,Yaxshilandi,False,3.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2015 entries, 0 to 2014
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bemor_id        2015 non-null   object 
 1   ism             2015 non-null   object 
 2   yosh            2015 non-null   int64  
 3   jinsi           2015 non-null   object 
 4   viloyat         1990 non-null   object 
 5   bo_lim          2015 non-null   object 
 6   shifokor        2015 non-null   object 
 7   tashxis         2015 non-null   object 
 8   qabul_sanasi    2015 non-null   object 
 9   yotish_kunlari  1955 non-null   float64
 10  narx_sum        1985 non-null   float64
 11  tolov_turi      2015 non-null   object 
 12  holat           2015 non-null   object 
 13  qayta_murojaat  2015 non-null   bool   
 14  reyting         1975 non-null   float64
dtypes: bool(1), float64(3), int64(1), object(10)
memory usage: 222.5+ KB


- `qabul_sanasi` ustuni str formatida ekan , uni datetime qilish kerak
- `qayta_murojaat` booleanda berilgan , uni int qilishimiz kerak

In [4]:
df.shape

(2015, 15)

In [5]:
df.describe()

,yosh,yotish_kunlari,narx_sum,reyting
count,2015.000000,1955.000000,1.985000e+03,1975.000000
mean,41.184615,14.325831,2.533063e+06,3.852152
std,23.978902,8.649002,1.392165e+06,1.159697
min,1.000000,0.000000,1.500370e+05,1.000000
25%,20.000000,7.000000,1.340064e+06,3.000000
50%,40.000000,14.000000,2.493145e+06,4.000000
75%,62.000000,22.000000,3.731877e+06,5.000000
max,84.000000,29.000000,4.999666e+06,5.000000


- be'morlarning o'rtachasi 41 yosh ekan . Eng keksasi 84 yoshgacha bo'lsa , eng kichigi 1 yosh .
- be'morlar o'rtacha 14 kun davomida qolib davolanishgan . Ularning aksari (75% izi) 22 kun davomida yotishgan . Qolgan 25% esa bir hafta yotib davolanishgan .
- klinikada maksimal ravishda ko'p qolganlar - 5 milliongacha , o'rtacha esa 2.5 milliongacha to'lov qilishgan
- berilgan reytinglardan - be'morlarning klinikaga nisbatan bergan ballarini ko'rish mumkin . Be'morlarning 50% izi 4 reytingni bergan , bundan - ularning kamida yarmi klinikaga 4 va 5 baho bergan , bu yaxshi natija .

In [6]:
df.isnull().sum()

,0
bemor_id,0
ism,0
yosh,0
jinsi,0
viloyat,25
bo_lim,0
shifokor,0
tashxis,0
qabul_sanasi,0
yotish_kunlari,60


- viloyat, yotish_kunlari, narx_sum va reyting ustunlarida tushib qolgan qiymatlar bor , ularni oldini olish kerak

In [7]:
df.duplicated().sum()

np.int64(15)

- datasetimizda bir xil ma'lumotlardan bir nechta borligi aniq bo'ldi

In [8]:
df['tashxis'].value_counts()

,count
tashxis,
Bronxit,110
Angina,87
Qalqonsimon bez,81
Travma,77
Osteoxondroz,76
Neyropatiya,74
Mioma,70
Ovarian kista,68
Semirishlik,67


- bemorlarda turli kasalliklar qayd etilgan . Buni tashhis natijalaridan ham bilib olsak bo'ladi

In [9]:
df['jinsi'].value_counts()

,count
jinsi,
Ayol,1065
Erkak,950


In [10]:
df['jinsi'].isnull().sum()

np.int64(0)

In [11]:
df['bo_lim'].value_counts()

,count
bo_lim,
Endokrinologiya,270
Pediatriya,263
Nevrologiya,260
Ginekologiya,256
Ortopediya,253
Jarrohlik,246
Terapiya,241
Kardiologiya,226


In [12]:
# bo_lim ustunini nomini o'zgartirish
df = df.rename(columns={'bo_lim':"bo'lim"})
df.columns

Index(['bemor_id', 'ism', 'yosh', 'jinsi', 'viloyat', 'bo'lim', 'shifokor',
       'tashxis', 'qabul_sanasi', 'yotish_kunlari', 'narx_sum', 'tolov_turi',
       'holat', 'qayta_murojaat', 'reyting'],
      dtype='object')

In [13]:
df.describe(include='object')

,bemor_id,ism,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,tolov_turi,holat
count,2015,2015,2015,1990,2015,2015,2015,2015,2015,2015
unique,2000,2000,2,10,8,18,31,926,4,4
top,P01888,Bemor_1888,Ayol,Navoiy,Endokrinologiya,Dr. Tursunova G,Bronxit,2023-07-14,Karta,Yaxshilandi
freq,2,2,1065,224,270,143,110,7,835,1187


- ushbu matnli ma'lumotlardan : viloyat ustunida bir qancha qiymatlar yoq'ligi ma'lum bo'lyapti
- bemor_id va ism ustunlarida 2000 ta ma'lumotlar takrorlanmas ma'lumotlardir
- idsi `P01888` ma'lumoti , qolganlarnikidan ko'p takrorlangan

#### 2. **Ma'lumotlarni tozalash**

In [14]:
df.shape

(2015, 15)

In [15]:
df.duplicated().sum()

np.int64(15)

- datasetimizda jami 15 ta duplikatlar bor ekan , ularni o'chiramiz

In [16]:
# takroriy ma'lumotlarni o'chirish
df = df.drop_duplicates()
df.shape

(2000, 15)

In [17]:
# null qiymatlarni tekshirish
df.isnull().sum()

,0
bemor_id,0
ism,0
yosh,0
jinsi,0
viloyat,25
bo'lim,0
shifokor,0
tashxis,0
qabul_sanasi,0
yotish_kunlari,60


In [18]:
df['viloyat'].value_counts()

,count
viloyat,
Navoiy,221
Qashqadaryo,220
Farg'ona,215
Samarqand,197
Andijon,193
Toshkent,191
Xorazm,190
Surxondaryo,189
Buxoro,186


In [19]:
# qaysi viloyatdanligi yozilmay qolganlarni "nomalum" bilan to'ldirish
df['viloyat'] = df['viloyat'].fillna("Noma'lum")
df['viloyat'].isnull().sum()

np.int64(0)

- yotish kunlari ustunida esa 60 ta ma'lumot yo'q . Ya'ni ko'p be'morlarni necha kun yotib davolangani noma'lum .
- yo'q qiymatlarni tashxis ustuniga qarab , uning o'rtacha qiymati bilan to'ldiramiz

In [20]:
df['yotish_kunlari'] = df.groupby('tashxis')['yotish_kunlari'].transform(lambda x: x.fillna(x.median()))
df['yotish_kunlari'].isnull().sum()

np.int64(0)

- `narx_sum` ustunida 30 ta ma'lumotlar yo'q , ya'ni 30 ta bemorning qanchaga davolangani yozilmagan .
- davolanish narxi qancha bo'lgani - be'morning qaysi bo'limda davolanganiga yoki qaysi shifokor ko'rigidan o'tganiga bog'liq bo'ladi . Shuning uchun bo'limga qarab o'rtacha qiymat bilan to'ldiramiz

In [21]:
df['narx_sum'] = df.groupby("bo'lim")['narx_sum'].transform(lambda x: x.fillna(x.mean()))
df['narx_sum'].isnull().sum()

np.int64(0)

In [22]:
df.head()

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
0,P00001,Bemor_1,52,Ayol,Farg'ona,Nevrologiya,Dr. Rahimov S,Migren,2022-02-20,14.0,2999095.0,Karta,Og'ir,True,1.0
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,1465502.0,Naqd,Surunkali,True,3.0
2,P00003,Bemor_3,72,Erkak,Surxondaryo,Pediatriya,Dr. Tursunova G,Ich ketish,2023-03-28,23.0,4530088.0,Naqd,Surunkali,False,3.0
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,1349110.0,Sug'urta,Yaxshilandi,True,4.0
4,P00005,Bemor_5,21,Erkak,Farg'ona,Jarrohlik,Dr. Abdullayev K,O'tkir appenditsit,2024-01-02,7.0,996462.0,Naqd,Yaxshilandi,False,3.0


- aksar be'morlar klinikaga yoki shifokorga ball bermagan . Lekin yaxshi doktrlarga ko'pincha yuqori ball berishadi . Shundan kelib chiqib har bir doktrga berilgan o'rtacha ball bilan NaN larni to'ldiramiz

In [23]:
df['reyting'] = df.groupby('shifokor')['reyting'].transform(lambda x: x.fillna(x.mean()))
df['reyting'].isnull().sum()

np.int64(0)

In [24]:
df.isnull().sum()

,0
bemor_id,0
ism,0
yosh,0
jinsi,0
viloyat,0
bo'lim,0
shifokor,0
tashxis,0
qabul_sanasi,0
yotish_kunlari,0


#####**Ma'lumot turlarini o'zgartirish**

In [25]:
df['qabul_sanasi'] = pd.to_datetime(df['qabul_sanasi'])
df['qabul_sanasi'].dtype

dtype('<M8[ns]')

- `qayta_murojaat` ustuni booleanda berilgan . Uni int8 turiga o'zgartiramiz . Shunda True lar 1 bo'ladi , False lar esa 0 bo'ladi
- bundan qayta_murojaat=1 bo'lsa , be'mor klinikaga avval ham kelgan bo'ladi , agarda 0 ga teng bo'lsa aksincha

In [26]:
df['qayta_murojaat'] = df['qayta_murojaat'].astype(np.int8)
df['qayta_murojaat'].head()

,qayta_murojaat
0,1
1,1
2,0
3,1
4,0


In [27]:
# takrorlangan qatorlar qolmaganini tekshirish
df.duplicated().sum()

np.int64(0)

In [28]:
df.shape

(2000, 15)

In [29]:
# qayta indekslab qo'yish
df = df.reset_index(drop=True)

In [30]:
df.head()

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
0,P00001,Bemor_1,52,Ayol,Farg'ona,Nevrologiya,Dr. Rahimov S,Migren,2022-02-20,14.0,2999095.0,Karta,Og'ir,1,1.0
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,1465502.0,Naqd,Surunkali,1,3.0
2,P00003,Bemor_3,72,Erkak,Surxondaryo,Pediatriya,Dr. Tursunova G,Ich ketish,2023-03-28,23.0,4530088.0,Naqd,Surunkali,0,3.0
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,1349110.0,Sug'urta,Yaxshilandi,1,4.0
4,P00005,Bemor_5,21,Erkak,Farg'ona,Jarrohlik,Dr. Abdullayev K,O'tkir appenditsit,2024-01-02,7.0,996462.0,Naqd,Yaxshilandi,0,3.0


####3. **Ma'lumotlarni Filtrlash va O'zgartirish**

In [31]:
# Kardialogiya bo'limidagi yoshi 50 dan yuqori be'morlarni ma'lumotlarini ko'rish
df.loc[(df["bo'lim"]=='Kardiologiya')&(df['yosh']>50)]

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
20,P00021,Bemor_21,76,Ayol,Navoiy,Kardiologiya,Dr. Yusupova M,Yurak yetishmovchiligi,2024-10-01,28.0,4255810.0,Karta,Yaxshilandi,1,3.0
35,P00036,Bemor_36,64,Erkak,Xorazm,Kardiologiya,Dr. Karimov A,Stenokardiya,2023-09-23,3.0,3525896.0,Karta,Surunkali,0,2.0
44,P00045,Bemor_45,60,Erkak,Qashqadaryo,Kardiologiya,Dr. Yusupova M,Aritmiya,2024-05-18,4.0,632450.0,Naqd,Yaxshilandi,0,5.0
70,P00071,Bemor_71,74,Erkak,Buxoro,Kardiologiya,Dr. Karimov A,Aritmiya,2023-09-11,5.0,4541649.0,Naqd,Yaxshilandi,1,4.0
112,P00113,Bemor_113,62,Erkak,Navoiy,Kardiologiya,Dr. Toshmatov B,Aritmiya,2024-01-10,29.0,179092.0,Naqd,Og'ir,0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1915,P01916,Bemor_1916,54,Ayol,Navoiy,Kardiologiya,Dr. Toshmatov B,Stenokardiya,2024-07-23,18.0,2648002.0,Sug'urta,Yaxshilandi,0,5.0
1918,P01919,Bemor_1919,65,Ayol,Xorazm,Kardiologiya,Dr. Yusupova M,Gipertenziya,2024-03-11,5.0,1724255.0,Sug'urta,Yaxshilandi,0,5.0
1965,P01966,Bemor_1966,51,Ayol,Samarqand,Kardiologiya,Dr. Yusupova M,Stenokardiya,2023-11-01,27.0,2683922.0,Sug'urta,Yaxshilandi,0,5.0
1967,P01968,Bemor_1968,68,Erkak,Xorazm,Kardiologiya,Dr. Toshmatov B,Aritmiya,2024-09-19,27.0,235350.0,Naqd,Yaxshilandi,1,4.0


In [32]:
# 1-10 qatorli ma'lumotlarni va be'morlarni ismi, yoshi va bo'limini ko'rish
df.iloc[:10, [1, 2, 5]]

,ism,yosh,bo'lim
0,Bemor_1,52,Nevrologiya
1,Bemor_2,15,Kardiologiya
2,Bemor_3,72,Pediatriya
3,Bemor_4,61,Jarrohlik
4,Bemor_5,21,Jarrohlik
5,Bemor_6,83,Terapiya
6,Bemor_7,75,Nevrologiya
7,Bemor_8,75,Nevrologiya
8,Bemor_9,24,Ortopediya
9,Bemor_10,3,Kardiologiya


In [33]:
# narxi bir milliondan yuqori hamda holati yaxshi bo'lganlarni ma'lumotlarini ko'rish
df.query("narx_sum>1000000 and holat=='Yaxshilandi'")

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,1349110.0,Sug'urta,Yaxshilandi,1,4.0
10,P00011,Bemor_11,22,Ayol,Surxondaryo,Kardiologiya,Dr. Yusupova M,Aritmiya,2023-03-31,20.0,1652718.0,Sug'urta,Yaxshilandi,0,5.0
11,P00012,Bemor_12,53,Ayol,Surxondaryo,Nevrologiya,Dr. Holiqova N,Neyropatiya,2023-07-24,5.0,2978340.0,Karta,Yaxshilandi,1,2.0
12,P00013,Bemor_13,2,Ayol,Buxoro,Jarrohlik,Dr. Abdullayev K,Travma,2023-02-05,0.0,1344737.0,Naqd,Yaxshilandi,1,5.0
13,P00014,Bemor_14,30,Erkak,Qashqadaryo,Jarrohlik,Dr. Xasanov R,Travma,2022-02-16,3.0,3693525.0,Naqd,Yaxshilandi,0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1986,P01987,Bemor_1987,77,Erkak,Qashqadaryo,Pediatriya,Dr. Tursunova G,ARVI,2022-06-08,7.0,1800882.0,Naqd,Yaxshilandi,1,3.0
1987,P01988,Bemor_1988,4,Erkak,Navoiy,Kardiologiya,Dr. Toshmatov B,Aritmiya,2022-01-23,9.0,2609150.0,Naqd,Yaxshilandi,0,3.0
1988,P01989,Bemor_1989,75,Erkak,Navoiy,Terapiya,Dr. Mirzayev J,Gastrit,2023-07-14,21.0,2175216.0,Naqd,Yaxshilandi,1,3.0
1991,P01992,Bemor_1992,39,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-01-04,7.0,2981455.0,Naqd,Yaxshilandi,0,5.0


In [34]:
# naqt to'lagan va qayta murojat qilgan be'morlar ma'lumotlari
df.query("tolov_turi=='Naqd' and qayta_murojaat==1")

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,1465502.0,Naqd,Surunkali,1,3.0
8,P00009,Bemor_9,24,Ayol,Farg'ona,Ortopediya,Dr. Normatov U,Osteoxondroz,2023-02-04,15.0,1769814.0,Naqd,Og'ir,1,4.0
12,P00013,Bemor_13,2,Ayol,Buxoro,Jarrohlik,Dr. Abdullayev K,Travma,2023-02-05,0.0,1344737.0,Naqd,Yaxshilandi,1,5.0
14,P00015,Bemor_15,38,Erkak,Toshkent,Kardiologiya,Dr. Karimov A,Gipertenziya,2022-06-21,10.0,548825.0,Naqd,Yaxshilandi,1,3.0
30,P00031,Bemor_31,62,Erkak,Toshkent,Ortopediya,Dr. Botirov A,Sinish,2024-02-03,22.0,4734751.0,Naqd,Surunkali,1,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1978,P01979,Bemor_1979,35,Erkak,Navoiy,Ginekologiya,Dr. Ismoilova Z,Mioma,2022-02-06,9.0,705134.0,Naqd,Yaxshilandi,1,2.0
1979,P01980,Bemor_1980,8,Ayol,Navoiy,Ginekologiya,Dr. Ismoilova Z,Mioma,2022-07-25,27.0,4297036.0,Naqd,Yaxshilandi,1,4.0
1986,P01987,Bemor_1987,77,Erkak,Qashqadaryo,Pediatriya,Dr. Tursunova G,ARVI,2022-06-08,7.0,1800882.0,Naqd,Yaxshilandi,1,3.0
1988,P01989,Bemor_1989,75,Erkak,Navoiy,Terapiya,Dr. Mirzayev J,Gastrit,2023-07-14,21.0,2175216.0,Naqd,Yaxshilandi,1,3.0


- 0 dan 18 yoshgacha -> "bola" , 19-40 o'rta , 41-60 katta , 61+ keksa qilib kategoriyalaymiz

In [35]:
def yosh_toifasi(yosh):
  if yosh <= 18:
    return 'Bola'
  elif yosh <= 40:
    return "O'rta"
  elif yosh <= 60:
    return "Katta"
  else:
    return "Keksa"

df['yosh'].apply(yosh_toifasi)

,yosh
0,Katta
1,Bola
2,Keksa
3,Keksa
4,O'rta
...,...
1995,Keksa
1996,Katta
1997,Katta
1998,Keksa


In [36]:
df['yosh'].describe()

,yosh
count,2000.000000
mean,41.185000
std,23.980165
min,1.000000
25%,20.000000
50%,40.000000
75%,62.000000
max,84.000000


In [37]:
# yuqoridagi kategoriyalashni pd.cut bilan - guruhlemiz
bins = [1, 18, 40, 60, 90]
group_bins = ["Bola", "O'rta", 'Katta', 'Keksa']
df['yosh_guruhlari'] = pd.cut(df['yosh'], bins=bins, labels=group_bins)
df[['yosh', 'yosh_guruhlari']].sample(10)

,yosh,yosh_guruhlari
1687,62,Keksa
971,27,O'rta
1488,30,O'rta
1704,12,Bola
988,5,Bola
895,65,Keksa
1964,75,Keksa
1523,17,Bola
385,63,Keksa
1526,22,O'rta


- be'morni to'lov miqdoridan kelib chiqib kategoriyaga ajratamiz : 1.5 mln so'mgacha "arzon" , 1.5mln-2.5mln o'rta va 3mln+ qimmat qilamiz

In [38]:
def narx_toifasi(narx):
  if narx <= 1500_000:
    return "Arzon"
  elif narx <= 2500_000:
    return "O'rta"
  else:
    return "Qimmat"

df['tolov_guruhlari'] = df['narx_sum'].apply(narx_toifasi)

In [39]:
df[['narx_sum', 'tolov_guruhlari']].sample(10)

,narx_sum,tolov_guruhlari
343,3108963.0,Qimmat
521,834659.0,Arzon
333,3571103.0,Qimmat
899,636598.0,Arzon
340,4028016.0,Qimmat
565,437906.0,Arzon
510,3700985.0,Qimmat
964,1005656.0,Arzon
1618,796004.0,Arzon
206,3487013.0,Qimmat


- holatlarni raqamga almashtirish : bunda yaxshilandi=3, surunkali=2, og'ir=1, kiritik=0 bo'ladi

In [40]:
df['holat'].value_counts()

,count
holat,
Yaxshilandi,1178
Surunkali,492
Og'ir,241
Kritik,89


In [41]:
holat_dct = {'Yaxshilandi':3, 'Surunkali':2, "Og'ir":1, 'Kritik':0}
df["holat_turi"] = df['holat'].map(holat_dct)
df[['holat', 'holat_turi']].head(10)

,holat,holat_turi
0,Og'ir,1
1,Surunkali,2
2,Surunkali,2
3,Yaxshilandi,3
4,Yaxshilandi,3
5,Yaxshilandi,3
6,Kritik,0
7,Surunkali,2
8,Og'ir,1
9,Surunkali,2


In [42]:
df['tolov_turi'].shape

(2000,)

In [43]:
df['tolov_turi'].value_counts()

,count
tolov_turi,
Karta,830
Naqd,711
Sug'urta,378
Kredit,81


- to'lov turlarini alohida ustun qilish

In [44]:
tolov_encoded = pd.get_dummies(df['tolov_turi'], prefix='tolov', dtype=int)
tolov_encoded.head(10)

,tolov_Karta,tolov_Kredit,tolov_Naqd,tolov_Sug'urta
0,1,0,0,0
1,0,0,1,0
2,0,0,1,0
3,0,0,0,1
4,0,0,1,0
5,0,0,0,1
6,0,0,0,1
7,0,0,0,1
8,0,0,1,0
9,0,0,0,1


In [45]:
df.shape

(2000, 18)

- tolov_encoded dataframeni asosiy jadvalga qo'shamiz

In [46]:
# ustun qilib qo'shamiz
df = pd.concat([df, tolov_encoded], axis=1)
df.head(10)

,bemor_id,ism,yosh,jinsi,viloyat,bo'lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,...,holat,qayta_murojaat,reyting,yosh_guruhlari,tolov_guruhlari,holat_turi,tolov_Karta,tolov_Kredit,tolov_Naqd,tolov_Sug'urta
0,P00001,Bemor_1,52,Ayol,Farg'ona,Nevrologiya,Dr. Rahimov S,Migren,2022-02-20,14.0,...,Og'ir,1,1.0,Katta,Qimmat,1,1,0,0,0
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,...,Surunkali,1,3.0,Bola,Arzon,2,0,0,1,0
2,P00003,Bemor_3,72,Erkak,Surxondaryo,Pediatriya,Dr. Tursunova G,Ich ketish,2023-03-28,23.0,...,Surunkali,0,3.0,Keksa,Qimmat,2,0,0,1,0
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,...,Yaxshilandi,1,4.0,Keksa,Arzon,3,0,0,0,1
4,P00005,Bemor_5,21,Erkak,Farg'ona,Jarrohlik,Dr. Abdullayev K,O'tkir appenditsit,2024-01-02,7.0,...,Yaxshilandi,0,3.0,O'rta,Arzon,3,0,0,1,0
5,P00006,Bemor_6,83,Ayol,Samarqand,Terapiya,Dr. Nazarova D,Gastrit,2022-07-09,27.0,...,Yaxshilandi,1,1.0,Keksa,Arzon,3,0,0,0,1
6,P00007,Bemor_7,75,Erkak,Qashqadaryo,Nevrologiya,Dr. Rahimov S,Epilepsiya,2023-10-31,2.0,...,Kritik,1,1.0,Keksa,Qimmat,0,0,0,0,1
7,P00008,Bemor_8,75,Erkak,Namangan,Nevrologiya,Dr. Holiqova N,Insult,2022-04-23,14.0,...,Surunkali,0,5.0,Keksa,Qimmat,2,0,0,0,1
8,P00009,Bemor_9,24,Ayol,Farg'ona,Ortopediya,Dr. Normatov U,Osteoxondroz,2023-02-04,15.0,...,Og'ir,1,4.0,O'rta,O'rta,1,0,0,1,0
9,P00010,Bemor_10,3,Ayol,Navoiy,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-12-14,3.0,...,Surunkali,0,4.0,Bola,O'rta,2,0,0,0,1


In [47]:
df.columns

Index(['bemor_id', 'ism', 'yosh', 'jinsi', 'viloyat', 'bo'lim', 'shifokor',
       'tashxis', 'qabul_sanasi', 'yotish_kunlari', 'narx_sum', 'tolov_turi',
       'holat', 'qayta_murojaat', 'reyting', 'yosh_guruhlari',
       'tolov_guruhlari', 'holat_turi', 'tolov_Karta', 'tolov_Kredit',
       'tolov_Naqd', 'tolov_Sug'urta'],
      dtype='object')

- asl `tolov_turi` ustuni keyingi tahlillarda kerak bo'ladi

####4. **Guruhlash va Agregatsiya**

- har bir bo'limdagi bemorlar soni, o'rtacha davolanish narxi va o'rtacha yotish kunlarini aniqlash

In [48]:
df.groupby("bo'lim").agg(
    bemor_soni=('bemor_id', 'nunique'),
    davolanish_narxi=('narx_sum', 'mean'),
    yotish_kuni=('yotish_kunlari', 'mean')
).sort_values(by='bemor_soni', ascending=False)

,bemor_soni,davolanish_narxi,yotish_kuni
bo'lim,,,
Endokrinologiya,268,2.533438e+06,14.664179
Pediatriya,263,2.515634e+06,13.125475
Nevrologiya,260,2.602848e+06,13.755769
Ginekologiya,253,2.438866e+06,14.492095
Ortopediya,249,2.593435e+06,14.654618
Jarrohlik,244,2.500075e+06,13.672131
Terapiya,239,2.543800e+06,14.374477
Kardiologiya,224,2.549458e+06,16.102679


- eng ko'p bemorlar - Endokrinologiyada ekan va ular 268 tani tashkil qilgan . Endokrinologiya va Pediatriyalarga marketing va reklamalar doim urg'u berib borishlari kerak .
- eng kami esa Kardiologiyada . Nima uchunligini aniqlash kerak : balki shifokorlar yetishmayotgandur .
- Nevrologiya bemorlar soni bo'yicha 3-o'rinda tursada ammo o'rtacha davolanish narxi qolganlarnikidan yuqoriroq
- barcha bo'limdagi bemorlarni necha kun yotib davolanganlari esa deyarli bir xil va davolanish narxi ham uncha katta farq qilmaydi

In [49]:
# shifokorlar sonini aniqlash
df['shifokor'].nunique()

18

In [50]:
# reyting ustunini butun son ko'rinishiga o'tkazish
df['reyting'] = df['reyting'].round(0)

df['reyting'] = df['reyting'].astype('int8')
df['reyting'].head()

,reyting
0,1
1,3
2,3
3,4
4,3


In [51]:
# shifokorlarni o'rtacha reytingini aniqlab top5 tasini olish
df.groupby('shifokor')['reyting'].mean().sort_values(ascending=False).head()

,reyting
shifokor,
Dr. Tursunova G,4.055944
Dr. Mamadaliyeva S,3.961538
Dr. Ismoilova Z,3.943089
Dr. Normatov U,3.931298
Dr. Nazarova D,3.922078


In [52]:
# har yili klinakaga nechta bemor kelib xizmatlardan foydalanganini aniqlash
df.groupby(df['qabul_sanasi'].dt.year)['bemor_id'].nunique().sort_index()

,bemor_id
qabul_sanasi,
2022,658
2023,634
2024,708


- 2022-yilda klinikada 658 ta tashrif bo'lgan bo'lsa, 2023-yilda bu ko'rsatkich 634 taga tushgan . Bunga raqobatchilarning ko'paygani, shifokorlar almashgani yoki marketing yaxshi bo'lmagani sabab bo'lgan bo'lishi mumkin
- 2023-yilda natijalar pastlaganiga qaramasdan yangi yilda kliniki eng katta natijaga erishgan

In [53]:
df["bo'lim"].value_counts()

,count
bo'lim,
Endokrinologiya,268
Pediatriya,263
Nevrologiya,260
Ginekologiya,253
Ortopediya,249
Jarrohlik,244
Terapiya,239
Kardiologiya,224


In [54]:
df.pivot_table(
    index="bo'lim",
    columns='jinsi',
    values='narx_sum',
    aggfunc='mean'
).sort_values(by='Erkak', ascending=False)

jinsi,Ayol,Erkak
bo'lim,,
Nevrologiya,2.596139e+06,2.610093e+06
Jarrohlik,2.397867e+06,2.595793e+06
Ortopediya,2.656654e+06,2.523252e+06
Kardiologiya,2.581909e+06,2.513975e+06
Endokrinologiya,2.593190e+06,2.456323e+06
Pediatriya,2.627443e+06,2.394064e+06
Terapiya,2.685582e+06,2.374702e+06
Ginekologiya,2.577543e+06,2.275083e+06


- erkaklar o'rtacha Nevrologiya va Jarrohlikga qolgan bo'limlarga qaraganda ko'proq xarajat qilganlar . Ayollar esa aksincha bu doktrlarga deyarli ehtiyoj sezmaganlar
- ayollar esa Terapiya va Ortopediyaga o'rtacha 2.5 mlndan ko'proq xarajat qilganlar . Bu klinika rahbariyatiga marketing kampaniyalarida aynan Terapiya va Pediatriya kompleks xizmatlarini ayollarga ko'proq taklif qilish foydaliroq ekanini ko'rsatadi
- Terapiya va Ortopediya bo'limlarida ayollar ko'proq daromad keltirayotgan bo'lsa, Jarrohlik bo'limida erkaklar o'rtacha ko'proq xarajat qilmoqda
- Ginekologiya - faqat ayollar muammolari bilan shug'ullanadigan bo'lim. Erkak bemorlarning bu bo'limda paydo bo'lib qolishi ma'lumotlar bazasida xatolik borligini bildiradi. Balki ba'zi ayol bemorlarning jinsi adashib "Erkak" deb kiritilgan yoki bo'lim nomi noto'g'ri yozilgan bo'lishi mumkiin

In [60]:
pd.crosstab(index=df['yosh_guruhlari'], columns=df['holat']).sort_values(by="Og'ir", ascending=False)

holat,Kritik,Og'ir,Surunkali,Yaxshilandi
yosh_guruhlari,,,,
O'rta,21,74,137,315
Keksa,27,57,124,323
Katta,20,56,117,274
Bola,17,52,106,248


- klinikaga kelayotgan o'rta va keksa yoshli insonlarning sog'lig'i juda qaltis holatda bo'ladi. Reanimatsiya va tez tibbiy yordam bo'limlari aynan mana shu ikki yosh guruhiga xizmat ko'rsatish uchun har doim shay turishi kerak
- yana o'sha O'rta yoshli va keksalarda surunkali kasalliklar qolgan yosh toifalariga qaraganda ko'proq . O'rta yoshlilarda 137 ta bo'lsa , keksalarda 124 ta qayd etilgan . Ular uchun uzoq muddatli davolanish paketlari, oilaviy cheklar yoki obunalar tashkil qilinsa, klinika o'zining doimiy daromad oqimini kafolatlaydi
- har bir yosh guruhidagi eng katta qiymatlar Yaxshilandi bo'limida turibdi . Bu klinika shifokorlarining muvaffaqiyat ko'rsatkichi (Success Rate) juda yuqori ekanligini anglatadi. Klinikaga kelgan bemorlarning yarmidan ko'pi u yerga kelib shifo topmoqda